# 06 · Local Signal Lab

**Purpose.** Исследовать Local как tactical signal через proxy-таргеты (RUONIA-keyrate spread) и проверить circularity прежней идеи m1_ruonia_mad_score.

**What to look for.**
- распределение и автокорреляция текущего Local
- circularity: m1_ruonia_mad_score vs |spread| (contemp) против forward-направления
- бьёт ли что-нибудь persistence на forward level/change T+1/T+7
- что честно можно/нельзя доказать на текущих данных

> Это lab-ноутбук: выводы здесь предварительные, не финальный отчёт. Меняй параметры в ячейке *Parameters* и перезапускай.

In [ ]:
# --- bootstrap: запуск из корня проекта (рядом с data/ и backend/) ---
import sys, os
from pathlib import Path
# найти корень проекта и встать в него
_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / 'data' / 'processed').is_dir()), _here)
os.chdir(_root)
sys.path.insert(0, str(_root))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 60)
from lab import utils as u
print('project root:', _root.name)

## Parameters
Меняй значения здесь и перезапускай ноутбук.

In [ ]:
HORIZONS = [1, 7]
ROLL_WINDOW = 90

In [ ]:
d = u.load_final_dataset()
saved = u.load_lsi_scores()
loc = saved[saved['lsi_local'].notna()].copy()
print('Local non-null:', len(loc), '| window', loc['date'].min().date() if len(loc) else None, '->', loc['date'].max().date() if len(loc) else None)

### Текущий Local: распределение, доля high-score, автокорреляция

In [ ]:
if len(loc):
    s = loc['lsi_local']
    print('share >=60:', round(float((s>=60).mean()),3), '| share >=30:', round(float((s>=30).mean()),3))
    for lag in [1,5,10,20]:
        print(f'autocorr lag {lag}:', round(float(s.autocorr(lag)),3))
    fig, axes = plt.subplots(1,2,figsize=(12,3.5))
    axes[0].hist(s, bins=30, color='teal', alpha=0.8); axes[0].set_title('Local LSI hist')
    axes[1].plot(loc['date'], s, lw=0.9); axes[1].set_title('Local LSI over window'); plt.show()

### Local vs Global в общем окне

In [ ]:
if len(loc):
    sh = saved[saved['lsi_local'].notna()]
    print('Spearman(Local, Global) shared window:', round(u.spearman(sh['lsi_local'], sh['lsi_global']),4))

### Дневной proxy: RUONIA − keyrate spread

In [ ]:
prox = u.build_ruonia_keyrate_proxy()
prox = u.add_forward_targets(prox, horizons=HORIZONS)
prox['spread_pctl'] = prox['spread'].rolling(ROLL_WINDOW, min_periods=30).apply(
    lambda w: (np.argsort(np.argsort(w))[-1]+1)/len(w), raw=True)
print('proxy rows', len(prox), '| spread nonnull', int(prox['spread'].notna().sum()))
prox[['date','ruonia_rate','key_rate','spread']].describe()

### Прикрепляем фичи-предикторы к дневному proxy (ffill)

In [ ]:
pred_feats = ['m1_ruonia_mad_score','m1_spread_mad_score','m5_cbr_liquidity_stress_mad_score']
prox = u.attach_feature_to_proxy(prox, d, pred_feats)
prox['comp_m1m5'] = ((prox['m1_ruonia_mad_score']-prox['m1_ruonia_mad_score'].mean())/prox['m1_ruonia_mad_score'].std()
    + (prox['m5_cbr_liquidity_stress_mad_score']-prox['m5_cbr_liquidity_stress_mad_score'].mean())/prox['m5_cbr_liquidity_stress_mad_score'].std())
prox.shape

### CIRCULARITY CHECK: m1_ruonia_mad_score
Сравни: корреляция с **|spread|** (контемпоральная, почти тавтология) против корреляции с **forward change** (настоящее предсказание направления).

In [ ]:
rows = []
rows.append({'test':'contemp vs |spread|','spearman':round(u.spearman(prox['m1_ruonia_mad_score'], prox['spread'].abs()),4)})
rows.append({'test':'contemp vs signed spread','spearman':round(u.spearman(prox['m1_ruonia_mad_score'], prox['spread']),4)})
for h in HORIZONS:
    rows.append({'test':f'forward change T+{h}','spearman':round(u.spearman(prox['m1_ruonia_mad_score'], prox[f'fwd_change_{h}']),4)})
    rows.append({'test':f'forward level T+{h}','spearman':round(u.spearman(prox['m1_ruonia_mad_score'], prox[f'fwd_level_{h}']),4)})
pd.DataFrame(rows)

### Baseline-сравнение предикторов forward spread

In [ ]:
preds = {'m1_ruonia_mad':'m1_ruonia_mad_score','m1_spread_mad':'m1_spread_mad_score',
         'm5_cbr_stress':'m5_cbr_liquidity_stress_mad_score','composite_m1m5':'comp_m1m5',
         'rolling_pctl':'spread_pctl','persistence(spread_t)':'spread'}
out = []
for h in HORIZONS:
    for name,col in preds.items():
        out.append({'horizon':f'T+{h}','predictor':name,
            'Sp_level':round(u.spearman(prox[col], prox[f'fwd_level_{h}']),4),
            'Sp_change':round(u.spearman(prox[col], prox[f'fwd_change_{h}']),4)})
pd.DataFrame(out).pivot_table(index='predictor', columns='horizon', values=['Sp_level','Sp_change'])

## Notes / Open questions

- Если forward-change корреляции ~0 у всех — directional Local на текущих фичах не доказывается.
- persistence обычно доминирует forward level — это и есть честный baseline.
- m1_ruonia_mad коррелирует с |spread| (магнитудой), а не с направлением — это circular.
- Для настоящего intraday Local нужны дневные данные денежного рынка (repo, o/n объёмы, аукционы ЦБ).